# CoChem-SCRIBE - Verified-Output Scientific Rendering & Provenance Pipeline

This notebook invokes genuine `cochem_scribe` verified-output APIs directly:
- 19-entry in-scope figure source register verification (`verify_source_register`)
- Scope exclusion audit (`audit_farir_exclusion`)
- Verified figure SHA-256 manifest verification (`sha256_file` against `benchmarks/verified_figures/manifest_sha256.json`)
- Provider status inspection (`scribe_provider`)

All dependencies and upstream artifacts fail closed when absent; synthetic random matrices and fabricated SCF energies are strictly prohibited.

In [ ]:
import json
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
repo_root = cwd.parent if cwd.name == "notebooks" else cwd
src_dir = repo_root / "src"
if src_dir.is_dir() and str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

# Fail closed if cochem_scribe or its required dependencies are unavailable
from cochem_scribe import (
    __version__,
    audit_farir_exclusion,
    render_geom_comparison,
    render_kinetic_rxn_profile,
    scribe_provider,
    sha256_file,
    verify_source_register,
)

print(f"[CoChem-SCRIBE v{__version__}] Loaded verified-output provider: {scribe_provider}")


## 1. 19/19 Figure Source Register & Scope Exclusion Verification

In [ ]:
manifest_path = repo_root / "manifests" / "in_scope_figure_register.json"
if not manifest_path.is_file():
    raise FileNotFoundError(f"Missing required figure register manifest: {manifest_path}")

register_report = verify_source_register(manifest_path)
audit_report = audit_farir_exclusion(manifest_path, raise_on_violation=True)
print("Register verification:", json.dumps(register_report, indent=2))
print("Exclusion audit status:", audit_report.get("status", "PASSED"))


## 2. Verified Figures SHA-256 Integrity Check

In [ ]:
figures_dir = repo_root / "benchmarks" / "verified_figures"
sha_manifest_path = figures_dir / "manifest_sha256.json"
if not sha_manifest_path.is_file():
    raise FileNotFoundError(f"Missing verified figure SHA-256 manifest: {sha_manifest_path}")

sha_manifest = json.loads(sha_manifest_path.read_text(encoding="utf-8"))
files_sha256 = sha_manifest["files_sha256"]
for filename, expected_hash in sorted(files_sha256.items()):
    artifact_path = figures_dir / filename
    actual_hash = sha256_file(artifact_path)
    if actual_hash != expected_hash:
        raise ValueError(f"SHA-256 mismatch for {filename}: expected {expected_hash}, got {actual_hash}")
    print(f"VERIFIED {filename}: {actual_hash}")
